# Experiment: CyEmbed Archetype Embedding Analysis

Interpretation and QC notebook for one saved run from `01_archetype_embedding_sweep.ipynb`.

Sections A-I:
- A. Reconstruction QC
- B. Archetype profiles
- C. Per-cell weights
- D. Sample-level summaries
- E. Cluster-level summaries
- F. UMAP overlays
- G. Marker embedding analysis
- H. Archetype embedding analysis
- I. Residual analysis


In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
  html {
    overflow-anchor: auto;
  }
  .jp-Notebook,
  .jp-Cell,
  .jp-OutputArea,
  .output_wrapper,
  .output {
    overflow-anchor: none;
  }
</style>
<script>
(() => {
  if (window.__cyembedScrollGuardInstalled) return;
  window.__cyembedScrollGuardInstalled = true;

  let lastScrollY = window.scrollY;
  let suppressUntil = 0;

  const rememberScroll = () => {
    if (Date.now() > suppressUntil) {
      lastScrollY = window.scrollY;
    }
  };

  window.addEventListener('scroll', rememberScroll, { passive: true });

  const observer = new MutationObserver(() => {
    const now = Date.now();
    const delta = Math.abs(window.scrollY - lastScrollY);
    if (delta > 120) {
      suppressUntil = now + 250;
      window.scrollTo(0, lastScrollY);
    }
  });

  observer.observe(document.body, { childList: true, subtree: true });
})();
</script>
"""))


In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from CyEmbed.analysis import (
    archetype_marker_rankings,
    cosine_similarity_matrix,
    dominant_assignments,
    kl_history_columns,
    load_run_outputs,
    nearest_neighbors_from_similarity,
    pairwise_distance_matrix,
    pca_projection,
    per_marker_reconstruction_stats,
    posterior_mean_weights,
    purity_summary,
    residual_norms,
    residual_summary,
    summarize_by_group,
    umap_projection,
    weight_entropy,
)
from CyEmbed.plotting import (
    plot_embedding_scatter,
    plot_clustermap,
    plot_matrix_heatmap,
    plot_observed_vs_reconstructed,
    plot_umap_categorical,
    plot_umap_overlay,
    plot_weight_histograms,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


sns.set_theme(context="notebook", style="whitegrid")


In [ ]:
# === Editable configuration ===
SWEEP_ROOT = Path("outputs/mcf7_probabilistic_archetype_embedding_sweep")
RUN_DIR = None  # Optional explicit run dir path
RUN_ID = None   # Optional explicit run_id (used when RUN_DIR is None)

ANALYSIS_OUTDIR = Path("outputs/mcf7_probabilistic_archetype_embedding_analysis")
ANALYSIS_OUTDIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_CFG = {
    "top_n_markers": 12,
    "scatter_markers": None,  # None -> pick top markers by R2
    "row_zscore_archetype_heatmap": True,
    "max_umap_panels": 12,
    "marker_umap_n_neighbors": 15,
    "marker_umap_min_dist": 0.3,
    "marker_umap_random_state": 7,
    "save_analysis_adata": True,
    "support_threshold_scales": [0.1, 0.25, 0.5],
    "practical_zero_threshold_scale": 0.25,

    # Run discovery/selection while sweep is still running
    # Options: "composite" or any column from runs_df (e.g. "val_recon")
    "run_selection_metric": "composite",
    "selected_k": 6,  # None -> allow all K; int or list -> filter candidate runs
    "selected_decoder_type": "factorized",  # None -> allow all; "factorized" or "direct"
    "archetype_usage_low_scale": 0.5,
    "archetype_dominance_low_scale": 0.5,
    "archetype_low_variance_quantile": 0.25,
    "require_no_dead_archetypes": True,
    "dead_archetype_threshold": 0,
    "composite_weights": {
        "val_recon": 1.0,
        "dead_archetypes": 2.0,
        "usage_std": 0.5,
        "mean_entropy": 0.25,
        "purity": 0.25,
    },

    # Optional AnnData path for UMAP overlays / conditions
    "adata_path": "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/data/MCF7.h5ad",
    "condition_col": None,
}


In [ ]:
# === Load run outputs ===
required_files = [
    "config.json",
    "summary_metrics.json",
    "history.csv",
    "X_observed.npy",
    "X_hat.npy",
    "W.npy",
    "A_hat.npy",
    "marker_names.csv",
    "cell_ids.csv",
]


def _discover_completed_runs(sweep_root: Path) -> pd.DataFrame:
    run_records = []
    if not sweep_root.exists():
        return pd.DataFrame()

    for run_dir in sorted([d for d in sweep_root.iterdir() if d.is_dir()]):
        if not all((run_dir / f).exists() for f in required_files):
            continue

        try:
            with (run_dir / "config.json").open("r", encoding="utf-8") as f:
                cfg = json.load(f)
            with (run_dir / "summary_metrics.json").open("r", encoding="utf-8") as f:
                sm = json.load(f)
            history = pd.read_csv(run_dir / "history.csv")
        except Exception as exc:
            print(f"Skipping unreadable run folder {run_dir.name}: {exc}")
            continue

        val_summary = sm.get("val", {})
        val_recon = sm.get("best_val_recon", sm.get("final_val_recon", np.nan))
        run_records.append(
            {
                "run_id": sm.get("run_id", run_dir.name),
                "run_dir": str(run_dir.resolve()),
                "model_type": cfg.get("model_type", "deterministic"),
                "decoder_type": cfg.get("decoder_type"),
                "use_residual_latent": cfg.get("use_residual_latent", False),
                "K": cfg.get("K"),
                "d": cfg.get("d"),
                "hidden_dims": cfg.get("hidden_dims"),
                "lr": cfg.get("lr"),
                "batch_size": cfg.get("batch_size"),
                "recon_loss_type": cfg.get("recon_loss_type"),
                "tau": cfg.get("tau"),
                "patience": cfg.get("patience"),
                "beta_w": cfg.get("beta_w", np.nan),
                "beta_r": cfg.get("beta_r", np.nan),
                "residual_dim": cfg.get("residual_dim", np.nan),
                "best_epoch": sm.get("best_epoch", np.nan),
                "stopped_early": sm.get("stopped_early", np.nan),
                "val_recon": val_recon,
                "final_val_recon": sm.get("final_val_recon", np.nan),
                "final_kl_w": sm.get("final_kl_w", np.nan),
                "final_kl_r": sm.get("final_kl_r", np.nan),
                "dead_archetypes": val_summary.get("dead_archetypes_lt_1pct", np.nan),
                "usage_std": val_summary.get("usage_std", np.nan),
                "mean_entropy": val_summary.get("mean_weight_entropy", np.nan),
                "purity_gt_08": val_summary.get("dominant_frac_gt_0_8", np.nan),
                "n_epochs_completed": int(history["epoch"].max()) if "epoch" in history.columns and len(history) > 0 else 0,
            }
        )

    if not run_records:
        return pd.DataFrame()

    runs_df = pd.DataFrame(run_records)
    return runs_df


def _add_composite_score(runs_df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    w = cfg["composite_weights"]
    out = runs_df.copy()
    out["composite_score"] = (
        w["val_recon"] * out["val_recon"].astype(float)
        + w["dead_archetypes"] * out["dead_archetypes"].astype(float)
        + w["usage_std"] * out["usage_std"].astype(float)
        + w["mean_entropy"] * out["mean_entropy"].astype(float)
        - w["purity"] * out["purity_gt_08"].astype(float)
    )
    return out


runs_df = _discover_completed_runs(SWEEP_ROOT)
if runs_df.empty:
    raise FileNotFoundError(
        f"No completed runs found in {SWEEP_ROOT}. "
        "Start the sweep notebook first, or wait for at least one run to finish."
    )

runs_df = _add_composite_score(runs_df, ANALYSIS_CFG)

eligible_df = runs_df.copy()
selected_k = ANALYSIS_CFG.get("selected_k")
if selected_k is not None:
    if isinstance(selected_k, (list, tuple, set, np.ndarray)):
        selected_k_values = [int(k) for k in selected_k]
        eligible_df = eligible_df.loc[eligible_df["K"].isin(selected_k_values)].copy()
    else:
        selected_k_values = [int(selected_k)]
        eligible_df = eligible_df.loc[eligible_df["K"] == selected_k_values[0]].copy()
    if eligible_df.empty:
        raise ValueError(f"No completed runs found for selected_k={selected_k_values}.")
else:
    selected_k_values = None
selected_decoder_type = ANALYSIS_CFG.get("selected_decoder_type")
if selected_decoder_type is not None:
    selected_decoder_type = str(selected_decoder_type).lower()
    if selected_decoder_type not in {"factorized", "direct"}:
        raise ValueError("selected_decoder_type must be one of {None, 'factorized', 'direct'}." )
    eligible_df = eligible_df.loc[eligible_df["decoder_type"].astype(str).str.lower() == selected_decoder_type].copy()
    if eligible_df.empty:
        raise ValueError(
            f"No completed runs found for selected_k={selected_k_values} and selected_decoder_type={selected_decoder_type}."
        )
base_eligible_df = eligible_df.copy()
if ANALYSIS_CFG.get("require_no_dead_archetypes", True):
    thr = ANALYSIS_CFG.get("dead_archetype_threshold", 0)
    filtered = eligible_df.loc[eligible_df["dead_archetypes"].fillna(np.inf) <= thr].copy()
    if not filtered.empty:
        eligible_df = filtered
    else:
        print("Warning: no runs satisfy the no-dead-archetype constraint; falling back to pre-filtered completed runs.")
        eligible_df = base_eligible_df.copy()

metric = ANALYSIS_CFG.get("run_selection_metric", "composite")
if metric == "composite":
    sort_col = "composite_score"
else:
    sort_col = metric
    if sort_col not in eligible_df.columns:
        raise ValueError(f"run_selection_metric={metric} is not a valid runs_df column.")
runs_ranked = eligible_df.sort_values(sort_col, ascending=True).reset_index(drop=True)
runs_snapshot_path = ANALYSIS_OUTDIR / "available_runs_snapshot.csv"
runs_ranked.to_csv(runs_snapshot_path, index=False)
print(f"Discovered completed runs: {len(runs_df)} | eligible for selection: {len(eligible_df)}")
print(f"Selected K filter: {selected_k_values if selected_k_values is not None else 'all'}")
print(f"Selected decoder filter: {selected_decoder_type if selected_decoder_type is not None else 'all'}")
print(f"Selection metric: {sort_col}")
print(f"Saved ranked run snapshot: {runs_snapshot_path}")
display(runs_ranked)

if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR)
elif RUN_ID is not None:
   
    selected = runs_ranked.loc[runs_ranked["run_id"].astype(str) == str(RUN_ID)]
    if selected.empty:
        raise ValueError(f"RUN_ID={RUN_ID} was not found among ranked runs.")
    RUN_DIR = Path(selected.iloc[0]["run_dir"])
else:
    RUN_DIR = Path(runs_ranked.iloc[0]["run_dir"])

run = load_run_outputs(RUN_DIR)
X = run["X_observed"]
X_hat = run["X_hat"]
marker_names = run["marker_names"]
cell_ids = run["cell_ids"]

W_sample = run.get("W_sample", run.get("W"))
if run.get("W_mean") is not None:
    W_mean = run["W_mean"]
elif run.get("mu_w") is not None:
    tau_val = float(run["config"].get("tau", 1.0))
    W_mean = posterior_mean_weights(run["mu_w"], tau=tau_val)
else:
    W_mean = None
W = W_mean if W_mean is not None else W_sample
W_label = "W_mean" if W_mean is not None else "W_sample"
available_weight_views = [name for name, arr in [("W_mean", W_mean), ("W_sample", W_sample)] if arr is not None]

A_hat = run["A_hat"]
sample_ids = run.get("sample_ids")
cluster_ids = run.get("cluster_ids")

print(f"Selected run dir: {RUN_DIR}")
print(f"Model type: {run['config'].get('model_type', 'deterministic')}")
print(f"Decoder: {run['config']['decoder_type']}")
print(f"Use residual latent: {run['config'].get('use_residual_latent', False)}")
print(f"X shape: {X.shape}")
print(f"W shape (analysis default: {W_label}): {W.shape}")
print(f"Available weight views: {available_weight_views}")
print(f"A_hat shape: {A_hat.shape}")

history_df = run["history"]
per_marker_df = per_marker_reconstruction_stats(X, X_hat, marker_names)
dom_df = dominant_assignments(W, cell_ids=cell_ids)
purity_df = purity_summary(W)

display(per_marker_df.head(15))
display(purity_df)


In [ ]:
# === Unified AnnData object + marker-space UMAP ===
import anndata as ad

default_weight_label = globals().get("W_label", "W")
archetype_names = [f"archetype_{k}" for k in range(W.shape[1])]

obs_df = pd.DataFrame(index=pd.Index(cell_ids, name="cell_id"))
obs_df["dominant_archetype"] = pd.Categorical(dom_df["dominant_archetype"].astype(str).to_numpy())
obs_df["dominant_weight"] = dom_df["dominant_weight"].to_numpy()
obs_df["weight_entropy"] = dom_df["entropy"].to_numpy()
if sample_ids is not None:
    obs_df["sample_id"] = pd.Categorical(pd.Index(sample_ids).astype(str))
if cluster_ids is not None:
    obs_df["cluster_id"] = pd.Categorical(pd.Index(cluster_ids).astype(str))
for k, archetype_name in enumerate(archetype_names):
    obs_df[f"{default_weight_label}_{archetype_name}"] = W[:, k]

var_df = pd.DataFrame(index=pd.Index(marker_names, name="marker"))
marker_stats_df = per_marker_df.set_index("marker").reindex(marker_names)
for col in marker_stats_df.columns:
    var_df[col] = marker_stats_df[col].to_numpy()
for k, archetype_name in enumerate(archetype_names):
    var_df[f"A_hat_{archetype_name}"] = A_hat[k]

adata_analysis = ad.AnnData(X=np.asarray(X, dtype=np.float32), obs=obs_df, var=var_df)
adata_analysis.layers["X_reconstructed"] = np.asarray(X_hat, dtype=np.float32)
adata_analysis.layers["residual"] = np.asarray(X - X_hat, dtype=np.float32)
adata_analysis.varm["A_hat"] = np.asarray(A_hat.T, dtype=np.float32)
adata_analysis.obsm["W"] = np.asarray(W, dtype=np.float32)
adata_analysis.uns["archetype_names"] = archetype_names
adata_analysis.uns["default_weight_label"] = default_weight_label
adata_analysis.uns["run_dir"] = str(RUN_DIR)
adata_analysis.uns["run_config_json"] = json.dumps(run["config"], indent=2, sort_keys=True)
adata_analysis.uns["summary_metrics_json"] = json.dumps(run.get("summary_metrics", {}), indent=2, sort_keys=True)

for arr_name in ["W_mean", "W_sample", "U", "U_sample", "mu_w", "logvar_w", "R", "R_sample", "mu_r", "logvar_r"]:
    arr = run.get(arr_name)
    if isinstance(arr, np.ndarray) and arr.shape[0] == adata_analysis.n_obs:
        adata_analysis.obsm[arr_name] = np.asarray(arr, dtype=np.float32)

if ANALYSIS_CFG.get("adata_path") is not None:
    try:
        ref_adata = ad.read_h5ad(ANALYSIS_CFG["adata_path"])
        ref_obs = ref_adata.obs.copy()
        ref_obs.index = ref_obs.index.astype(str)
        ref_obs = ref_obs.add_prefix("source_")
        adata_analysis.obs = adata_analysis.obs.join(ref_obs.reindex(adata_analysis.obs_names))
        if "X_umap" in ref_adata.obsm:
            external_umap_df = pd.DataFrame(ref_adata.obsm["X_umap"], index=ref_adata.obs_names.astype(str))
            external_umap = external_umap_df.reindex(adata_analysis.obs_names).to_numpy(dtype=np.float32)
            if np.isfinite(external_umap).any():
                adata_analysis.obsm["X_umap_external"] = external_umap
    except Exception as exc:
        print(f"Unable to merge external AnnData metadata: {exc}")

marker_umap = umap_projection(
    X,
    n_neighbors=min(int(ANALYSIS_CFG.get("marker_umap_n_neighbors", 15)), max(2, X.shape[0] - 1)),
    min_dist=float(ANALYSIS_CFG.get("marker_umap_min_dist", 0.3)),
    random_state=int(ANALYSIS_CFG.get("marker_umap_random_state", 7)),
)
if marker_umap is None:
    print("umap-learn not installed; skipping marker-space UMAP.")
else:
    adata_analysis.obsm["X_umap"] = np.asarray(marker_umap, dtype=np.float32)
    adata_analysis.obsm["X_umap_markers"] = np.asarray(marker_umap, dtype=np.float32)
    plot_umap_categorical(
        adata_analysis.obsm["X_umap_markers"],
        adata_analysis.obs["dominant_archetype"].astype(str).to_numpy(),
        title="Marker-space UMAP colored by dominant archetype",
    )

analysis_adata_path = ANALYSIS_OUTDIR / f"{RUN_DIR.name}_analysis.h5ad"
if ANALYSIS_CFG.get("save_analysis_adata", True):
    adata_analysis.write_h5ad(analysis_adata_path)
    print(f"Saved analysis AnnData: {analysis_adata_path}")
print(adata_analysis)


In [ ]:
# === Probabilistic diagnostics ===
is_probabilistic = str(run["config"].get("model_type", "deterministic")).lower() == "probabilistic"
if not is_probabilistic:
    print("Deterministic run: probabilistic diagnostics are not applicable.")
else:
    print("Probabilistic run detected.")

    # 1) Sampled-vs-posterior-mean simplex weights.
    if W_sample is not None and W_mean is not None:
        w_abs_diff = np.abs(W_sample - W_mean)
        print(f"Mean |W_sample - W_mean|: {w_abs_diff.mean():.6f}")
        print(f"Median |W_sample - W_mean|: {np.median(w_abs_diff):.6f}")

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.histplot(w_abs_diff.ravel(), bins=80, ax=ax, color="tab:purple")
        ax.set_title("|W_sample - W_mean| distribution")
        ax.set_xlabel("Absolute difference")
        plt.tight_layout()

    # 2) KL tracking during optimization.
    kl_cols = kl_history_columns(history_df)
    if kl_cols:
        fig, ax = plt.subplots(figsize=(7, 4))
        for col in kl_cols:
            ax.plot(history_df["epoch"], history_df[col], label=col)
        ax.set_title("KL terms across training")
        ax.set_xlabel("Epoch")
        ax.legend(frameon=False)
        plt.tight_layout()
    else:
        print("No KL columns found in history.")

    # 3) Posterior variance of archetype logits q(U|X): var = exp(logvar_w).
    if run.get("logvar_w") is not None:
        logvar_w = np.asarray(run["logvar_w"], dtype=np.float32)
        var_w = np.exp(np.clip(logvar_w, -20.0, 20.0))
        w_for_diag = np.asarray(W_mean if W_mean is not None else W, dtype=np.float32)
        usage = w_for_diag.mean(axis=0)
        dominant_idx = np.argmax(w_for_diag, axis=1)
        dominant_fraction = np.array([(dominant_idx == k).mean() for k in range(w_for_diag.shape[1])], dtype=np.float32)
        uniform_usage = 1.0 / max(w_for_diag.shape[1], 1)
        usage_low_threshold = float(ANALYSIS_CFG.get("archetype_usage_low_scale", 0.5)) * uniform_usage
        dominance_low_threshold = float(ANALYSIS_CFG.get("archetype_dominance_low_scale", 0.5)) * uniform_usage
        low_variance_quantile = float(ANALYSIS_CFG.get("archetype_low_variance_quantile", 0.25))

        print(f"logvar_w mean: {logvar_w.mean():.6f} | median: {np.median(logvar_w):.6f}")
        print(f"var_w mean: {var_w.mean():.6f} | median: {np.median(var_w):.6f}")

        per_arch_df = pd.DataFrame({
            "archetype": [f"A{k}" for k in range(logvar_w.shape[1])],
            "mean_logvar_w": logvar_w.mean(axis=0),
            "mean_var_w": var_w.mean(axis=0),
            "median_var_w": np.median(var_w, axis=0),
        })
        display(per_arch_df)

        low_variance_threshold = float(np.quantile(per_arch_df["mean_var_w"].to_numpy(), low_variance_quantile))
        archetype_diag_df = per_arch_df.copy()
        archetype_diag_df["mean_usage"] = usage
        archetype_diag_df["dominant_fraction"] = dominant_fraction
        archetype_diag_df["low_usage"] = archetype_diag_df["mean_usage"] < usage_low_threshold
        archetype_diag_df["low_dominance"] = archetype_diag_df["dominant_fraction"] < dominance_low_threshold
        archetype_diag_df["low_variance"] = archetype_diag_df["mean_var_w"] <= low_variance_threshold

        def _classify_archetype(row):
            if row["low_usage"] and row["low_dominance"]:
                return "near-dead"
            if row["low_variance"] and row["low_dominance"]:
                return "collapsed-helper"
            if row["low_variance"]:
                return "stable-active"
            if row["low_dominance"]:
                return "shared-variable"
            return "active-variable"

        archetype_diag_df["status"] = archetype_diag_df.apply(_classify_archetype, axis=1)
        print(
            f"Archetype diagnostic thresholds: usage < {usage_low_threshold:.6f} "
            f"({ANALYSIS_CFG.get('archetype_usage_low_scale', 0.5):g} x 1/K), "
            f"dominant_fraction < {dominance_low_threshold:.6f} "
            f"({ANALYSIS_CFG.get('archetype_dominance_low_scale', 0.5):g} x 1/K), "
            f"mean_var_w <= {low_variance_threshold:.6f} "
            f"(q={low_variance_quantile:.2f})"
        )
        display(archetype_diag_df[[
            "archetype",
            "mean_usage",
            "dominant_fraction",
            "mean_var_w",
            "median_var_w",
            "status",
        ]])

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(logvar_w.ravel(), bins=80, ax=axes[0], color="tab:blue")
        axes[0].set_title("logvar_w distribution")
        axes[0].set_xlabel("logvar_w")

        sns.histplot(var_w.ravel(), bins=80, ax=axes[1], color="tab:orange")
        axes[1].set_title("var_w = exp(logvar_w) distribution")
        axes[1].set_xlabel("var_w")
        plt.tight_layout()

        fig, ax = plt.subplots(figsize=(max(6, 0.7 * logvar_w.shape[1]), 3.8))
        sns.barplot(data=per_arch_df, x="archetype", y="mean_var_w", ax=ax, color="tab:orange")
        ax.set_title("Mean posterior logit variance per archetype")
        ax.set_ylabel("mean var_w")
        ax.set_xlabel("archetype")
        plt.tight_layout()
    else:
        print("No logvar_w saved for this run.")

    # 4) Optional residual latent diagnostics.
    if run.get("R") is not None or run.get("R_sample") is not None:
        r_for_norm = run.get("R") if run.get("R") is not None else run.get("R_sample")
        r_norm = residual_norms(r_for_norm)
        print(f"Residual norm mean: {r_norm.mean():.6f}")
        print(f"Residual norm median: {np.median(r_norm):.6f}")

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.histplot(r_norm, bins=80, ax=ax, color="tab:green")
        ax.set_title("Residual latent norm distribution")
        ax.set_xlabel("||R||")
        plt.tight_layout()

        if run.get("logvar_r") is not None:
            logvar_r = np.asarray(run["logvar_r"], dtype=np.float32)
            var_r = np.exp(np.clip(logvar_r, -20.0, 20.0))
            print(f"logvar_r mean: {logvar_r.mean():.6f} | median: {np.median(logvar_r):.6f}")
            print(f"var_r mean: {var_r.mean():.6f} | median: {np.median(var_r):.6f}")

            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(logvar_r.ravel(), bins=80, ax=axes[0], color="tab:cyan")
            axes[0].set_title("logvar_r distribution")
            axes[0].set_xlabel("logvar_r")

            sns.histplot(var_r.ravel(), bins=80, ax=axes[1], color="tab:red")
            axes[1].set_title("var_r = exp(logvar_r) distribution")
            axes[1].set_xlabel("var_r")
            plt.tight_layout()

        if sample_ids is not None:
            sample_resid = pd.DataFrame({"sample": sample_ids, "residual_norm": r_norm})
            sample_resid_summary = sample_resid.groupby("sample", dropna=False)["residual_norm"].mean().reset_index()
            display(sample_resid_summary.sort_values("residual_norm", ascending=False).head(20))

        if cluster_ids is not None:
            cluster_resid = pd.DataFrame({"cluster": cluster_ids, "residual_norm": r_norm})
            cluster_resid_summary = cluster_resid.groupby("cluster", dropna=False)["residual_norm"].mean().reset_index()
            display(cluster_resid_summary.sort_values("residual_norm", ascending=False).head(20))
    else:
        print("No residual latent arrays found for this run.")

    # Note: robust Var(W) requires multiple posterior samples per cell.
    if run.get("W_sample") is not None and run.get("W_mean") is not None:
        print("Tip: for stronger simplex-uncertainty estimates, evaluate with multiple MC samples and store per-cell Var(W).")


In [ ]:
# === A) Reconstruction QC ===
selected_markers = ANALYSIS_CFG["scatter_markers"]
if selected_markers is None:
    selected_markers = per_marker_df.head(min(6, len(marker_names)))["marker"].tolist()

plot_observed_vs_reconstructed(
    X,
    X_hat,
    marker_names=marker_names,
    markers=selected_markers,
    max_points=6000,
    random_state=7,
)

qc_heat = per_marker_df.set_index("marker")[["pearson_r", "spearman_r", "r2"]].T.to_numpy()
plot_matrix_heatmap(
    qc_heat,
    row_labels=["Pearson r", "Spearman r", "R2"],
    col_labels=per_marker_df["marker"].tolist(),
    title="Per-marker reconstruction quality",
    cmap="coolwarm",
    figsize=(max(12, len(marker_names) * 0.3), 3.8),
    vmin=-1,
    vmax=1,
)

display(per_marker_df.sort_values("r2", ascending=False).head(20))
display(per_marker_df.sort_values("r2", ascending=True).head(20))


In [ ]:
# === B) Archetype profiles ===
plot_matrix_heatmap(
    A_hat,
    row_labels=[f"A{k}" for k in range(A_hat.shape[0])],
    col_labels=marker_names,
    title="A_hat (archetype x marker)",
    cmap="viridis",
    figsize=(max(12, len(marker_names) * 0.35), max(4, A_hat.shape[0] * 0.35)),
)

if ANALYSIS_CFG["row_zscore_archetype_heatmap"]:
    row_mean = A_hat.mean(axis=1, keepdims=True)
    row_std = np.maximum(A_hat.std(axis=1, keepdims=True), 1e-8)
    A_row_z = (A_hat - row_mean) / row_std
    plot_matrix_heatmap(
        A_row_z,
        row_labels=[f"A{k}" for k in range(A_hat.shape[0])],
        col_labels=marker_names,
        title="A_hat row-wise z-score",
        cmap="coolwarm",
        figsize=(max(12, len(marker_names) * 0.35), max(4, A_hat.shape[0] * 0.35)),
        vmin=-2.5,
        vmax=2.5,
    )

ranking_df = archetype_marker_rankings(
    A_hat,
    marker_names=marker_names,
    top_n=int(ANALYSIS_CFG["top_n_markers"]),
)
ranking_path = ANALYSIS_OUTDIR / "archetype_marker_rankings.csv"
ranking_df.to_csv(ranking_path, index=False)
print(f"Saved marker ranking table to: {ranking_path}")

display(ranking_df.head(40))


In [ ]:
# === C) Per-cell weights ===
uniform_weight = 1.0 / max(W.shape[1], 1)
support_threshold_scales = np.asarray(ANALYSIS_CFG.get("support_threshold_scales", [0.1, 0.25, 0.5]), dtype=float)
support_thresholds = support_threshold_scales * uniform_weight
practical_zero_threshold_scale = float(ANALYSIS_CFG.get("practical_zero_threshold_scale", 0.25))
practical_zero_threshold = practical_zero_threshold_scale * uniform_weight
print(f"Adaptive support thresholds: {support_thresholds.tolist()} (uniform baseline=1/K={uniform_weight:.6f})")
print(f"Adaptive practical zero threshold: {practical_zero_threshold:.6f} ({practical_zero_threshold_scale:g} x 1/K)")

weight_views = []
if W_mean is not None:
    weight_views.append(("W_mean", W_mean))
if W_sample is not None:
    weight_views.append(("W_sample", W_sample))
if not weight_views:
    weight_views.append((W_label, W))

for label, W_view in weight_views:
    print(f"Weight histograms for {label}")
    plot_weight_histograms(W_view)

overview_rows = []
threshold_rows = []
dominant_count_frames = []
fig, axes = plt.subplots(len(weight_views), 4, figsize=(16, 4 * len(weight_views)), squeeze=False)
for row_idx, (label, W_view) in enumerate(weight_views):
    dom_view = dominant_assignments(W_view, cell_ids=cell_ids)
    purity_view = purity_summary(W_view)
    if "threshold" in purity_view.columns and "fraction_cells" in purity_view.columns:
        high_purity_target = 0.8
        high_purity_rows = purity_view.loc[np.isclose(purity_view["threshold"], high_purity_target), "fraction_cells"]
        high_purity_fraction = float(high_purity_rows.iloc[0]) if not high_purity_rows.empty else float(purity_view["fraction_cells"].max())
    else:
        high_purity_fraction = np.nan
    effective_support = np.exp(dom_view["entropy"].to_numpy())
    practical_support_size = (W_view > practical_zero_threshold).sum(axis=1)
    practical_zero_fraction = (W_view <= practical_zero_threshold).mean(axis=1)
    top2_mass = np.sort(W_view, axis=1)[:, -2:].sum(axis=1)

    overview_rows.append({
        "weights": label,
        "mean_effective_support": float(effective_support.mean()),
        "median_effective_support": float(np.median(effective_support)),
        "mean_top1": float(dom_view["dominant_weight"].mean()),
        "mean_top2_mass": float(top2_mass.mean()),
        "fraction_cells_top1_gt_0_8": high_purity_fraction,
    })
    for thr in support_thresholds:
        support_size_thr = (W_view > thr).sum(axis=1)
        zero_fraction_thr = (W_view <= thr).mean(axis=1)
        threshold_rows.append({
            "weights": label,
            "threshold": float(thr),
            "mean_support": float(support_size_thr.mean()),
            "median_support": float(np.median(support_size_thr)),
            "mean_zero_fraction": float(zero_fraction_thr.mean()),
            "median_zero_fraction": float(np.median(zero_fraction_thr)),
        })

    sns.histplot(dom_view["dominant_weight"], bins=50, ax=axes[row_idx, 0], color="tab:blue")
    axes[row_idx, 0].set_title(f"{label}: dominant weight")
    axes[row_idx, 0].set_xlabel("max_k W_ik")

    sns.histplot(effective_support, bins=min(50, max(10, W_view.shape[1] * 3)), ax=axes[row_idx, 1], color="tab:orange")
    axes[row_idx, 1].set_title(f"{label}: effective support")
    axes[row_idx, 1].set_xlabel("exp(entropy(W_i))")

    sns.histplot(
        practical_support_size,
        bins=np.arange(0.5, W_view.shape[1] + 1.5, 1.0),
        ax=axes[row_idx, 2],
        color="tab:green",
    )
    axes[row_idx, 2].set_title(f"{label}: support size")
    axes[row_idx, 2].set_xlabel(f"# archetypes with W_ik > {practical_zero_threshold:g}")

    sns.histplot(practical_zero_fraction, bins=30, ax=axes[row_idx, 3], color="tab:red")
    axes[row_idx, 3].set_title(f"{label}: practical zero fraction")
    axes[row_idx, 3].set_xlabel(f"fraction with W_ik <= {practical_zero_threshold:g}")

    dom_counts = dom_view["dominant_archetype"].value_counts().sort_index()
    dominant_count_frames.append(dom_counts.rename(label))

plt.tight_layout()
overview_df = pd.DataFrame(overview_rows)
threshold_summary_df = pd.DataFrame(threshold_rows)
display(overview_df)
display(threshold_summary_df)
dominant_counts_df = pd.concat(dominant_count_frames, axis=1).fillna(0).astype(int)
display(dominant_counts_df)
display(purity_df)


In [ ]:
# === D) Sample-level summaries ===
if sample_ids is None:
    print("No sample IDs found in saved outputs; skipping sample-level analysis.")
else:
    sample_stats = summarize_by_group(W, sample_ids, group_name="sample")
    sample_mean = sample_stats["mean_weights"]
    sample_dom = sample_stats["dominant_fractions"]

    display(sample_mean.head())
    display(sample_dom.head())

    sample_mean_matrix = sample_mean.set_index("sample").to_numpy()
    plot_matrix_heatmap(
        sample_mean_matrix,
        row_labels=sample_mean["sample"].astype(str).tolist(),
        col_labels=[c for c in sample_mean.columns if c != "sample"],
        title="Mean archetype weights per sample",
        cmap="magma",
        figsize=(8, max(4, sample_mean.shape[0] * 0.4)),
    )

    # Optional condition-level boxplots if adata + condition column provided.
    if ANALYSIS_CFG["adata_path"] is not None and ANALYSIS_CFG["condition_col"] is not None:
        import anndata as ad

        adata = ad.read_h5ad(ANALYSIS_CFG["adata_path"])
        obs = adata.obs.copy()
        if ANALYSIS_CFG["condition_col"] in obs.columns and "sample" in sample_mean.columns:
            map_df = obs[[ANALYSIS_CFG["condition_col"]]].copy()
            map_df["sample"] = obs.index.astype(str)
            # This assumes sample IDs are comparable to obs index or have been pre-mapped.
            print("Condition boxplots may require custom sample-to-condition mapping for your dataset.")


In [ ]:
# === E) Cluster-level summaries ===
if cluster_ids is None:
    print("No cluster IDs found in saved outputs; skipping cluster-level analysis.")
else:
    cluster_stats = summarize_by_group(W, cluster_ids, group_name="cluster")
    cluster_mean = cluster_stats["mean_weights"]
    cluster_dom = cluster_stats["dominant_fractions"]

    display(cluster_mean)
    display(cluster_dom)

    cluster_mean_matrix = cluster_mean.set_index("cluster").to_numpy()
    plot_matrix_heatmap(
        cluster_mean_matrix,
        row_labels=cluster_mean["cluster"].astype(str).tolist(),
        col_labels=[c for c in cluster_mean.columns if c != "cluster"],
        title="Mean archetype weights per cluster",
        cmap="plasma",
        figsize=(8, max(4, cluster_mean.shape[0] * 0.4)),
    )


In [ ]:
# === F) UMAP overlays ===
umap_xy = None
if ANALYSIS_CFG["adata_path"] is not None:
    try:
        import anndata as ad

        adata = ad.read_h5ad(ANALYSIS_CFG["adata_path"])
        if "X_umap" in adata.obsm:
            umap_df = pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names)
            # Align to run cell order if possible.
            common = [c for c in cell_ids if c in umap_df.index]
            if len(common) == len(cell_ids):
                umap_xy = umap_df.loc[cell_ids].to_numpy()
            elif len(common) > 0:
                print("Partial overlap between run cells and adata.obs_names; using matched subset only.")
                keep_mask = np.array([c in umap_df.index for c in cell_ids])
                umap_xy = umap_df.loc[np.array(cell_ids)[keep_mask]].to_numpy()
                W = W[keep_mask]
                dom_df = dom_df.loc[keep_mask].reset_index(drop=True)
            else:
                print("No matching cell IDs between run outputs and adata.obs_names.")
    except Exception as exc:
        print(f"Unable to load UMAP from AnnData: {exc}")

if umap_xy is None:
    print("UMAP unavailable (set ANALYSIS_CFG['adata_path'] with .obsm['X_umap']).")
else:
    n_panels = min(W.shape[1], int(ANALYSIS_CFG["max_umap_panels"]))
    n_cols = 4
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.3 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for k in range(n_panels):
        sc = axes[k].scatter(umap_xy[:, 0], umap_xy[:, 1], c=W[:, k], s=3, cmap="viridis", linewidths=0)
        axes[k].set_title(f"Archetype {k} weight")
        axes[k].set_xlabel("UMAP1")
        axes[k].set_ylabel("UMAP2")
        plt.colorbar(sc, ax=axes[k], fraction=0.046, pad=0.04)
    for j in range(n_panels, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()

    plot_umap_categorical(umap_xy, dom_df["dominant_archetype"].to_numpy(), title="Dominant archetype")
    plot_umap_overlay(umap_xy, dom_df["entropy"].to_numpy(), title="Entropy overlay", cmap="magma")


In [ ]:
# === G) Marker embedding analysis (E) ===
E = run.get("E")
if E is None:
    print("This run uses direct decoder; marker embedding matrix E is unavailable.")
else:
    E_pca = pca_projection(E, n_components=2)
    plot_embedding_scatter(E_pca, labels=marker_names, title="Marker embeddings PCA")

    E_umap = umap_projection(E, n_neighbors=min(10, max(2, E.shape[0] - 1)), min_dist=0.1, random_state=7)
    if E_umap is not None:
        plot_embedding_scatter(E_umap, labels=marker_names, title="Marker embeddings UMAP")
    else:
        print("umap-learn not installed; skipping marker embedding UMAP.")

    marker_sim = cosine_similarity_matrix(E)
    plot_matrix_heatmap(
        marker_sim,
        row_labels=marker_names,
        col_labels=marker_names,
        title="Marker embedding cosine similarity",
        cmap="coolwarm",
        figsize=(max(8, len(marker_names) * 0.3), max(6, len(marker_names) * 0.3)),
        vmin=-1,
        vmax=1,
    )

    plot_clustermap(
        marker_sim,
        row_labels=marker_names,
        col_labels=marker_names,
        title="Marker embedding clustermap (cosine similarity)",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        figsize=(max(7, len(marker_names) * 0.25), max(7, len(marker_names) * 0.25)),
        metric="euclidean",
        method="average",
    )

    nn_df = nearest_neighbors_from_similarity(marker_sim, marker_names, k=min(8, max(1, len(marker_names) - 1)))
    nn_path = ANALYSIS_OUTDIR / "marker_embedding_neighbors.csv"
    nn_df.to_csv(nn_path, index=False)
    print(f"Saved marker nearest-neighbor table to: {nn_path}")
    display(nn_df.head(40))


In [ ]:
# === H) Archetype embedding analysis (Z) ===
if run.get("Z") is not None:
    archetype_repr = run["Z"]
    repr_name = "Z"
else:
    archetype_repr = A_hat
    repr_name = "A_hat (direct archetype profiles)"

arch_sim = cosine_similarity_matrix(archetype_repr)
arch_dist = pairwise_distance_matrix(archetype_repr)

labels = [f"A{k}" for k in range(archetype_repr.shape[0])]
plot_matrix_heatmap(arch_sim, row_labels=labels, col_labels=labels, title=f"Archetype cosine similarity ({repr_name})", cmap="coolwarm", vmin=-1, vmax=1)
plot_matrix_heatmap(arch_dist, row_labels=labels, col_labels=labels, title=f"Archetype Euclidean distance ({repr_name})", cmap="viridis")

plot_clustermap(
    arch_sim,
    row_labels=labels,
    col_labels=labels,
    title=f"Archetype clustermap ({repr_name})",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    figsize=(6, 6),
    metric="euclidean",
    method="average",
)

arch_pca = pca_projection(archetype_repr, n_components=2)
plot_embedding_scatter(arch_pca, labels=labels, title=f"Archetype PCA ({repr_name})")


In [ ]:
# === I) Residual analysis ===
res_df = residual_summary(X, X_hat, marker_names)
res_path = ANALYSIS_OUTDIR / "residual_marker_summary.csv"
res_df.to_csv(res_path, index=False)
print(f"Saved residual summary to: {res_path}")

display(res_df.head(25))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot((X - X_hat).ravel(), bins=80, ax=axes[0], color="tab:blue")
axes[0].set_title("Global residual distribution")
axes[0].set_xlabel("Residual")

worst_markers = res_df.head(min(8, len(res_df)))["marker"].tolist()
idx_map = {m: i for i, m in enumerate(marker_names)}
for marker in worst_markers:
    m = idx_map[marker]
    sns.histplot((X[:, m] - X_hat[:, m]), bins=50, ax=axes[1], alpha=0.25, label=marker)
axes[1].set_title("Residuals for underfit markers")
axes[1].set_xlabel("Residual")
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()

# Optional residual heatmap by dominant archetype.
residual = X - X_hat
dom = np.argmax(W, axis=1)
res_by_dom = np.vstack([residual[dom == k].mean(axis=0) if np.any(dom == k) else np.zeros(X.shape[1]) for k in range(W.shape[1])])
plot_matrix_heatmap(
    res_by_dom,
    row_labels=[f"domA{k}" for k in range(W.shape[1])],
    col_labels=marker_names,
    title="Mean residual by dominant archetype",
    cmap="coolwarm",
    figsize=(max(12, len(marker_names) * 0.35), max(4, W.shape[1] * 0.3)),
)


In [ ]:
import scanpy as sc
sc.pl.umap(adata_analysis,color=['dominant_archetype'],
           cmap='seismic',vcenter=0,vmin='p01',vmax='p99',)

In [ ]:
sc.pl.umap(adata_analysis,color=['dominant_archetype']+list(adata_analysis.var_names),
           cmap='seismic',vcenter=0,vmin='p01',vmax='p99')

In [ ]:
adata_analysis